## Edge-level thread corpus descriptives

Coverage, text length, and training feasibility for the London undirected edge corpus.
The notebook reads from PostgreSQL; it does not modify the corpus table.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

CONNECTION_PATH = "/home/ilyesvirag/twitter/venv/connection.json"
TABLE = "interaction_edge_thread_corpus_london"

with open(CONNECTION_PATH, encoding="utf-8") as f:
    cfg = json.load(f)

engine = create_engine(URL.create(
    drivername="postgresql+psycopg2",
    username=cfg["user"], password=cfg["password"],
    host=cfg["host"], port=cfg["port"], database=cfg["database"]
), pool_pre_ping=True)

def query(sql):
    return pd.read_sql(text(sql), engine)


### 1. Overall corpus size

Includes raw and cleaned word counts; cleaned counts require the cleaning step.


In [ ]:
overall = query(f"""
SELECT
    COUNT(*) AS n_edges,
    COUNT(*) FILTER (WHERE n_threads > 1) AS n_edges_multiple_threads,
    COUNT(*) FILTER (WHERE n_thread_tweets > 1) AS n_edges_multiple_thread_tweets,
    SUM(n_interaction_tweets) AS total_interaction_tweets,
    SUM(n_threads) AS total_edge_thread_links,
    SUM(n_thread_tweets) AS total_thread_tweets,
    SUM(n_text_characters) AS total_text_characters,
    SUM(n_text_words) AS total_text_words,
    COUNT(edge_thread_cleaned) AS n_edges_with_cleaned_text,
    SUM(n_cleaned_words) AS total_cleaned_words,
    AVG(n_threads) AS mean_threads_per_edge,
    AVG(n_thread_tweets) AS mean_thread_tweets_per_edge,
    AVG(n_text_characters) AS mean_characters_per_edge,
    AVG(n_text_words) AS mean_words_per_edge,
    AVG(n_cleaned_words) AS mean_cleaned_words_per_edge
FROM {TABLE}
""")
overall.T

### 2. Edge-level distributions

In [ ]:
variables = ["n_interaction_tweets", "n_threads", "n_thread_tweets", "n_text_characters", "n_text_words", "n_cleaned_words"]
parts = []
for v in variables:
    x = query(f"""
    SELECT
        '{v}' AS variable,
        AVG({v}) AS mean,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY {v}) AS median,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY {v}) AS p75,
        PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY {v}) AS p90,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY {v}) AS p95,
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY {v}) AS p99,
        MAX({v}) AS max
    FROM {TABLE}
    """)
    parts.append(x)
distribution = pd.concat(parts, ignore_index=True)
distribution

### 3. How many edges have enough text?

In [ ]:
thresholds = query(f"""
WITH base AS (SELECT COUNT(*)::numeric AS n FROM {TABLE})
SELECT 'thread tweets' AS measure, x.threshold,
       COUNT(*) AS n_edges,
       COUNT(*) / base.n AS share_edges
FROM {TABLE}, base
CROSS JOIN (VALUES (1),(2),(5),(10),(20),(50),(100),(200)) AS x(threshold)
WHERE n_thread_tweets >= x.threshold
GROUP BY x.threshold, base.n
UNION ALL
SELECT 'raw words', x.threshold, COUNT(*), COUNT(*) / base.n
FROM {TABLE}, base
CROSS JOIN (VALUES (5), (10), (20), (50),(100),(250),(500),(1000),(2500),(5000),(10000)) AS x(threshold)
WHERE n_text_words >= x.threshold
GROUP BY x.threshold, base.n
UNION ALL
SELECT 'cleaned words', x.threshold, COUNT(*), COUNT(*) / base.n
FROM {TABLE}, base
CROSS JOIN (VALUES (5), (10), (20), (50),(100),(250),(500),(1000),(2500),(5000),(10000)) AS x(threshold)
WHERE edge_thread_cleaned IS NOT NULL AND n_cleaned_words >= x.threshold
GROUP BY x.threshold, base.n
ORDER BY measure, threshold
""")
thresholds

In [ ]:
for measure in ["thread tweets", "num of words"]:
    d = thresholds[thresholds["measure"] == measure]
    plt.figure(figsize=(7, 4))
    plt.plot(d["threshold"], d["share_edges"] * 100, marker="o")
    plt.xscale("log")
    plt.xlabel(f"Minimum {measure} per edge")
    plt.ylabel("Share of edges (%)")
    plt.title(f"Corpus coverage by minimum {measure}")
    plt.grid(alpha=0.2)
    plt.show()


### 4. Number of threads per edge

In [ ]:
thread_counts = query(f"""
SELECT
    CASE
        WHEN n_threads = 1 THEN '1'
        WHEN n_threads = 2 THEN '2'
        WHEN n_threads BETWEEN 3 AND 5 THEN '3-5'
        WHEN n_threads BETWEEN 6 AND 10 THEN '6-10'
        WHEN n_threads BETWEEN 11 AND 20 THEN '11-20'
        ELSE '21+'
    END AS n_threads_group,
    COUNT(*) AS n_edges
FROM {TABLE}
GROUP BY 1
""")
order = ['1','2','3-5','6-10','11-20','21+']
thread_counts['n_threads_group'] = pd.Categorical(thread_counts['n_threads_group'], categories=order, ordered=True)
thread_counts = thread_counts.sort_values('n_threads_group')
thread_counts['share_edges'] = thread_counts['n_edges'] / thread_counts['n_edges'].sum()
thread_counts

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(thread_counts['n_threads_group'].astype(str), thread_counts['share_edges'] * 100)
plt.xlabel('Number of threads per edge')
plt.ylabel('Share of edges (%)')
plt.title('Thread coverage across edges')
plt.show()


### 5. Concentration of cleaned text across edges

In [ ]:
concentration = query(f"""
WITH ranked AS (
    SELECT n_cleaned_words,
           NTILE(100) OVER (ORDER BY n_cleaned_words DESC) AS pct
    FROM {TABLE}
    WHERE edge_thread_cleaned IS NOT NULL
), totals AS (
    SELECT SUM(n_cleaned_words)::numeric AS total_words FROM ranked
)
SELECT
    SUM(n_cleaned_words) FILTER (WHERE pct <= 1) / total_words AS share_words_top_1pct_edges,
    SUM(n_cleaned_words) FILTER (WHERE pct <= 5) / total_words AS share_words_top_5pct_edges,
    SUM(n_cleaned_words) FILTER (WHERE pct <= 10) / total_words AS share_words_top_10pct_edges,
    SUM(n_cleaned_words) FILTER (WHERE pct <= 25) / total_words AS share_words_top_25pct_edges
FROM ranked, totals
GROUP BY total_words
""")
concentration.T

### 6. Relationship between interaction intensity and available text

The scatter plot and correlation table include cleaned word counts.


In [ ]:
sample = query(f"""
SELECT n_interaction_tweets, n_threads, n_thread_tweets, n_text_words, n_cleaned_words
FROM {TABLE}
TABLESAMPLE SYSTEM (1)
LIMIT 200000
""")
sample.corr(method='spearman')

In [ ]:
plot_sample = sample[(sample.n_interaction_tweets > 0) & (sample.n_cleaned_words > 0)].copy()
plt.figure(figsize=(7, 5))
plt.scatter(plot_sample['n_interaction_tweets'], plot_sample['n_cleaned_words'], s=8, alpha=0.15)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Interaction tweets per edge')
plt.ylabel('Cleaned words in edge corpus')
plt.title('Observed interaction intensity and available text')
plt.show()


### 7. Largest edge corpora

In [ ]:
largest = query(f"""
SELECT city, user_id1, user_id2, n_interaction_tweets, n_threads,
       n_thread_tweets, n_text_characters, n_text_words, n_cleaned_words
FROM {TABLE}
ORDER BY n_cleaned_words DESC
LIMIT 30
""")
largest

### 8. What to look for

The most important diagnostics are: (1) how many edges have only one thread or very little text; (2) how quickly the usable sample shrinks when requiring 5, 10, 20, 50, or 100 text tweets; (3) how concentrated the total text corpus is among a small share of edges; and (4) how strongly text availability is mechanically related to observed interaction intensity. These determine whether edge-level semantic/topic measures would be broadly usable or mainly describe a small set of heavily observed relationships.

### 9. Full-corpus BERTopic feasibility: settings and eligible population

The following diagnostics are designed for training on **all eligible edges**, rather than fitting on a sample and transforming the remainder. `n_cleaned_words >= 10` defines eligibility. The 500-word window is a *random contiguous window* per eligible edge; it does not increase the number of training documents. Estimates below are planning figures, not benchmarks of full-data UMAP/HDBSCAN.

The 500-word window is an optional pre-embedding sampling cap, not the transformer token limit. The MiniLM embedding model uses a maximum of 128 transformer tokens; actual truncation depends on tokenization.


In [ ]:
import numpy as np
from pathlib import Path

MIN_CLEANED_WORDS = 10
MAX_WORD_WINDOW = 500  # optional whitespace-word window, NOT the transformer token limit
MODEL_MAX_TOKENS = 128  # sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
EMBEDDING_DIM = 384  # paraphrase-multilingual-MiniLM-L12-v2; verified below if model loads
UMAP_COMPONENTS = 5
EMBEDDING_BATCH_SIZE = 64
BENCHMARK_ROWS = 2000
RUN_EMBEDDING_BENCHMARK = False  # Enable only when benchmarking is needed
RUN_TOKEN_LENGTH_CHECK = False  # Enable only when checking token lengths
RUN_EXACT_DUPLICATE_CHECK = False  # expensive on a large corpus


### 10. Cleaned-text availability, eligible edges, and training-text mass

In [ ]:
eligible = query(f"""
SELECT COUNT(*) AS all_edges,
       COUNT(*) FILTER (WHERE edge_thread_cleaned IS NOT NULL) AS edges_with_cleaned_text,
       COUNT(*) FILTER (WHERE n_cleaned_words IS NULL) AS missing_cleaned_word_counts,
       COUNT(*) FILTER (WHERE edge_thread_cleaned IS NOT NULL AND n_cleaned_words >= {MIN_CLEANED_WORDS}) AS eligible_edges,
       COUNT(*) FILTER (WHERE edge_thread_cleaned IS NOT NULL AND n_cleaned_words < {MIN_CLEANED_WORDS}) AS short_edges,
       SUM(n_cleaned_words) FILTER (WHERE edge_thread_cleaned IS NOT NULL AND n_cleaned_words >= {MIN_CLEANED_WORDS}) AS eligible_cleaned_words,
       SUM(LEAST(n_cleaned_words, {MAX_WORD_WINDOW})) FILTER (WHERE edge_thread_cleaned IS NOT NULL AND n_cleaned_words >= {MIN_CLEANED_WORDS}) AS windowed_words,
       COUNT(*) FILTER (WHERE edge_thread_cleaned IS NOT NULL AND n_cleaned_words > {MAX_WORD_WINDOW}) AS edges_above_word_window,
       SUM(n_interaction_tweets) FILTER (WHERE edge_thread_cleaned IS NOT NULL AND n_cleaned_words >= {MIN_CLEANED_WORDS}) AS eligible_interaction_tweets,
       SUM(n_threads) FILTER (WHERE edge_thread_cleaned IS NOT NULL AND n_cleaned_words >= {MIN_CLEANED_WORDS}) AS eligible_edge_thread_links
FROM {TABLE}
""")
e = eligible.iloc[0]
eligible['eligible_edge_share'] = eligible.eligible_edges / eligible.all_edges
eligible['share_above_word_window'] = eligible.edges_above_word_window / eligible.eligible_edges
eligible['word_window_share'] = eligible.windowed_words / eligible.eligible_cleaned_words
eligible.T


### 11. Cleaned-word length distribution and cutoff sensitivity

In [ ]:
length_buckets = query(f"""
SELECT CASE WHEN n_cleaned_words IS NULL THEN 'missing'
            WHEN n_cleaned_words < 10 THEN '0-9'
            WHEN n_cleaned_words < 20 THEN '10-19'
            WHEN n_cleaned_words < 50 THEN '20-49'
            WHEN n_cleaned_words < 100 THEN '50-99'
            WHEN n_cleaned_words < 250 THEN '100-249'
            WHEN n_cleaned_words < 500 THEN '250-499'
            WHEN n_cleaned_words < 1000 THEN '500-999'
            WHEN n_cleaned_words < 2500 THEN '1000-2499'
            ELSE '2500+' END AS length_group,
       COUNT(*) AS n_edges,
       SUM(n_cleaned_words) AS cleaned_words,
       SUM(LEAST(n_cleaned_words, {MAX_WORD_WINDOW})) AS capped_words
FROM {TABLE}
GROUP BY 1
""")
order = ['missing','0-9','10-19','20-49','50-99','100-249','250-499','500-999','1000-2499','2500+']
length_buckets['length_group'] = pd.Categorical(length_buckets.length_group, categories=order, ordered=True)
length_buckets = length_buckets.sort_values('length_group')
length_buckets['edge_share'] = length_buckets.n_edges / length_buckets.n_edges.sum()
length_buckets


In [ ]:
cutoffs = query(f"""
SELECT x.min_words,
       COUNT(*) FILTER (WHERE edge_thread_cleaned IS NOT NULL AND n_cleaned_words >= x.min_words) AS n_eligible,
       SUM(LEAST(n_cleaned_words, {MAX_WORD_WINDOW})) FILTER (WHERE edge_thread_cleaned IS NOT NULL AND n_cleaned_words >= x.min_words) AS windowed_words
FROM {TABLE}
CROSS JOIN (VALUES (5),(10),(20),(30),(50),(75),(100)) AS x(min_words)
GROUP BY x.min_words ORDER BY x.min_words
""")
cutoffs['share_all_edges'] = cutoffs.n_eligible / int(e.all_edges)
cutoffs['share_windowed_words_vs_min10'] = cutoffs.windowed_words / float(e.windowed_words)
cutoffs


### 12. Database storage and available machine resources

PostgreSQL relation size includes indexes and TOAST. Python RAM, free disk and GPU memory are measured on the machine running the notebook, which may differ from the PostgreSQL server.

In [ ]:
db_storage = query(f"""
SELECT pg_size_pretty(pg_relation_size('{TABLE}'::regclass)) AS table_heap,
       pg_size_pretty(pg_indexes_size('{TABLE}'::regclass)) AS indexes,
       pg_size_pretty(pg_total_relation_size('{TABLE}'::regclass)) AS total_with_toast_and_indexes,
       pg_total_relation_size('{TABLE}'::regclass) AS total_bytes
""")
db_storage.T


In [ ]:
import os, shutil
import psutil
resources = {'cpu_count_logical': psutil.cpu_count(), 'cpu_count_physical': psutil.cpu_count(logical=False),
             'ram_total_gb': psutil.virtual_memory().total / 1e9, 'ram_available_gb': psutil.virtual_memory().available / 1e9,
             'disk_free_gb': shutil.disk_usage('/mnt/common-hdd/group-dirs/twitter-2025-project').free / 1e9}
try:
    import torch
    resources['cuda_available'] = torch.cuda.is_available()
    if torch.cuda.is_available():
        resources['gpu_name'] = torch.cuda.get_device_name(0)
        free, total = torch.cuda.mem_get_info(0)
        resources['gpu_free_gb'], resources['gpu_total_gb'] = free / 1e9, total / 1e9
except ImportError:
    resources['cuda_available'] = 'torch not installed'
pd.Series(resources, name='value').to_frame()


### 13. Minimum storage estimates for all eligible embeddings

These are *lower bounds*: they exclude model objects, nearest-neighbor indices, UMAP/HDBSCAN working arrays, Python/Pandas overhead, text storage, and temporary copies. UMAP/HDBSCAN full-data peak RAM cannot be inferred from the embedding matrix alone.

In [ ]:
N = int(e.eligible_edges)
storage = pd.DataFrame([
    {'item': 'Sentence embeddings float32', 'dimensions': EMBEDDING_DIM, 'bytes_per_value': 4},
    {'item': 'Sentence embeddings float16 (disk only)', 'dimensions': EMBEDDING_DIM, 'bytes_per_value': 2},
    {'item': 'UMAP 5D float32', 'dimensions': UMAP_COMPONENTS, 'bytes_per_value': 4},
    {'item': 'Topic assignments int32', 'dimensions': 1, 'bytes_per_value': 4},
    {'item': 'One int64 row identifier', 'dimensions': 1, 'bytes_per_value': 8},
])
storage['eligible_edges'] = N
storage['estimated_gb_decimal'] = storage.dimensions * storage.bytes_per_value * N / 1e9
storage


### 14. Observed sampling and potential duplication

This diagnostic checks a small random page sample. It does **not** establish the global number of duplicate texts. Repeated thread context across different edges is expected and is a separate weighting issue.

In [ ]:
text_sample = query(f"""
SELECT city, user_id1, user_id2, n_cleaned_words, n_interaction_tweets, n_threads,
       edge_thread_cleaned
FROM {TABLE} TABLESAMPLE SYSTEM (1) REPEATABLE (12345)
WHERE edge_thread_cleaned IS NOT NULL AND n_cleaned_words >= {MIN_CLEANED_WORDS}
LIMIT {BENCHMARK_ROWS}
""")
print('Sample rows:', len(text_sample))
if len(text_sample):
    print('Exact duplicate cleaned-text share in this sample:', f'{text_sample.edge_thread_cleaned.duplicated(keep=False).mean():.1%}')
    display(text_sample[['n_cleaned_words','n_interaction_tweets','n_threads']].describe(percentiles=[.5,.9,.95,.99]))


### 15. Transformer token truncation: optional diagnostic

The multilingual MiniLM model used in the pilot has a 128-token maximum. A 500-word window can therefore lose substantial text **before** embedding. This optional check reports tokenizer lengths for sampled full cleaned texts and for their capped windows; it does not modify the planned model.

**Important:** The 500-word window is a separate optional preprocessing choice, not the model limit. The embedding model accepts at most 128 transformer tokens (including special tokens). Enable `RUN_TOKEN_LENGTH_CHECK` to measure actual truncation on the sampled cleaned texts.


In [ ]:
if RUN_TOKEN_LENGTH_CHECK and len(text_sample):
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    rng = np.random.default_rng(12345)
    def window(s):
        w = str(s).split()
        if len(w) <= MAX_WORD_WINDOW: return ' '.join(w)
        start = int(rng.integers(0, len(w) - MAX_WORD_WINDOW + 1))
        return ' '.join(w[start:start + MAX_WORD_WINDOW])
    model_texts = [window(s) for s in text_sample.edge_thread_cleaned]
    token_lengths = [len(tokenizer.encode(s, add_special_tokens=True, truncation=False)) for s in model_texts]
    display(pd.Series(token_lengths).describe(percentiles=[.5,.75,.9,.95,.99]).to_frame('transformer_tokens'))
    print('Share exceeding model token limit:', np.mean(np.array(token_lengths) > MODEL_MAX_TOKENS))
else:
    print('Skipped. Set RUN_TOKEN_LENGTH_CHECK=True to run.')


### 16. Embedding throughput: optional small benchmark

Benchmark the actual server/GPU before projecting runtime. Results include only embedding computation, not database I/O, UMAP or HDBSCAN.

In [ ]:
if RUN_EMBEDDING_BENCHMARK and len(text_sample):
    import time
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    # SentenceTransformer enforces the model token limit when encoding.
    texts = [' '.join(str(s).split()[:MAX_WORD_WINDOW]) for s in text_sample.edge_thread_cleaned]
    t0 = time.perf_counter()
    vectors = model.encode(texts, batch_size=EMBEDDING_BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True)
    seconds = time.perf_counter() - t0
    rate = len(texts) / seconds
    display(pd.Series({'benchmark_documents': len(texts), 'seconds': seconds, 'documents_per_second': rate,
                       'projected_embedding_hours': N / rate / 3600, 'actual_embedding_dimension': vectors.shape[1]}).to_frame('value'))
else:
    print('Skipped. Set RUN_EMBEDDING_BENCHMARK=True to run.')


### 17. Full-data training decision sheet

Review the figures above before launching: eligible N, embedding disk/RAM lower bounds, free disk and RAM, token truncation, benchmark throughput, and sample text duplication. **Full-data UMAP and HDBSCAN remain the major unresolved scalability risks.** An end-to-end trial at increasing sizes (for example 100k → 500k → 1m) should measure peak RAM and runtime before launching on all eligible edges. Topic count and outlier shares at those sizes can also guide whether `min_cluster_size=100` remains sensible. The `distributions(.20)` reassignment is deliberately excluded from the initial full-data fit.